In [ ]:
from whatstk import WhatsAppChat

In [ ]:
path = '../data/whatsapp_chat.txt'
chat = WhatsAppChat.from_source(filepath=path)


In [ ]:
chat.df.head()

In [ ]:
chat.df.info()

In [ ]:
chat.df.describe(include='all')

In [ ]:
chat.df['username'].unique()

In [ ]:
chat.df.to_csv('../data/whatsapp_chat.csv', index=False)

## Step 1: Clean the messages

The model learns to reproduce whatever is in the text, so we remove noise but keep personality.

**Removed:** system users (`You`, `GLOBAL TALENTS`), attachment-only messages (stickers, photos, audio), shared documents, and deleted messages.

**Fixed:** `<This message was edited>` tags stripped, invisible Unicode characters stripped (these wrapped the @mentions), links replaced with `<link>`, numeric WhatsApp IDs in mentions replaced with `@user`, curly quotes normalised, line breaks inside a message replaced with a space.

**Kept as-is:** casing, punctuation, Pidgin/slang, emojis, @mentions by name.

In [ ]:
import re

SYSTEM_USERS = ['You', 'GLOBAL TALENTS']

ATTACHED = re.compile(r'<attached: [^>]*>')
OMITTED = re.compile(r'\b(?:image|video|sticker|audio|GIF|video note|Contact card) omitted\s*$')
EDITED = re.compile(r'<This message was edited>')
DELETED = re.compile(r'^(?:This message was deleted.*|You deleted this message\.?)$')
INVISIBLE = re.compile('[\u200b-\u200f\u202a-\u202e\u2060-\u2069\ufeff]')
URL = re.compile(
    r'(?:https?:[/\\]{2}|www\.)\S+'
    r'|\b(?:[\w-]+\.)+(?:com|ng|io|ai|co|app|dev|org|net|click|ly)/\S*'
)
ID_MENTION = re.compile(r'@\d{6,}')
QUOTES = str.maketrans({'\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"'})


def clean_message(message):
    """Return the cleaned message, or '' if the message should be dropped."""
    if 'document omitted' in message:
        return ''
    message = INVISIBLE.sub('', message)
    message = ATTACHED.sub('', message)
    message = OMITTED.sub('', message)
    message = EDITED.sub('', message)
    message = URL.sub('<link>', message)
    message = ID_MENTION.sub('@user', message)
    message = message.translate(QUOTES)
    message = re.sub(r'\s+', ' ', message).strip()
    if DELETED.match(message):
        return ''
    return message

In [ ]:
df = chat.df.copy()
before = len(df)

df = df[~df['username'].isin(SYSTEM_USERS)]
df['message'] = df['message'].map(clean_message)
df = df[df['message'] != ''].reset_index(drop=True)

print(f'{before} -> {len(df)} messages ({before - len(df)} dropped)')
df['username'].value_counts()

In [ ]:
# Sanity check: every count except <link> should be 0 (or close to it)
for pattern in ['<attached', 'omitted', '<This message was edited>', 'This message was deleted', '\u2068', '\n', '<link>']:
    print(f'{pattern!r:32} {df["message"].str.contains(pattern, regex=False).sum()}')

df.sample(10)

In [ ]:
df.to_csv('../data/whatsapp_chat_clean.csv', index=False)